### Data Reading

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import Window

In [0]:
df_orders=spark.read.format("parquet")\
    .load("abfss://bronze@dbenttoend.dfs.core.windows.net/orders")

In [0]:
display(df_orders)

In [0]:
df_orders=df_orders.withColumnRenamed("_rescued_data","newschema")


In [0]:
df_orders=df_orders.drop("newschema")

In [0]:
df_orders.display()

In [0]:
df_orders=df_orders.withColumn("order_date",to_timestamp(col("order_date")))

In [0]:
df_orders.display()

In [0]:
df_orders=df_orders.withColumn("Year",year(col("order_date")))

### Calculating the ranks on total ammount

In [0]:
df_orders=df_orders.withColumn("row_flag",row_number().over(Window.partitionBy("Year").orderBy(col("total_amount").desc())))
df_orders.display()

### Classes - OOP

In [0]:
df_orders_new=df_orders

In [0]:

def dense_rank_flag(self, df_orders):
    return df_orders.withColumn(
        "dense_rank_flag",
        dense_rank().over(Window.partitionBy("Year").orderBy(col("total_amount").desc()))
    )

def rank_flag(self, df_orders):
    return df_orders.withColumn(
        "rank_flag",
        rank().over(Window.partitionBy("Year").orderBy(col("total_amount").desc()))
    )

def row_number_flag(self, df_orders):
    return df_orders.withColumn(
        "row_number_flag",
        row_number().over(Window.partitionBy("Year").orderBy(col("total_amount").desc()))
    )
        

### Datawriting


In [0]:
df_orders.write.format("delta")\
    .mode("append")\
        .save("abfss://silver@dbenttoend.dfs.core.windows.net/orders")